# CSAT & NPS – Deep Dive, Non-Imputed Voice Scoring, Validation & Customer-Month Data Model
**Customer Health & Sentiment Scoring Multi-Agent AI Solution**

| Part | What it does |
|---|---|
| **A. CSAT/NPS deep profile** | Coverage, distributions, missingness mechanism, drivers, consistency, incremental signal, segment bias, hygiene |
| **B. Text sentiment component** | Text score per interaction (plug-in from Sentiment Agent, or a baseline model trained on a customer-grouped split) |
| **C. Non-imputed scoring spec** | Observed-only re-weighting of Text / CSAT / NPS, survey cap, conflict rule, confidence, drivers, actions |
| **D. Customer-month data model** | `fact_interaction_scored` → `customer_month_voice` (+ segment NPS aggregate), data dictionary, imputation guard tests |
| **E. Validation framework** | 10 checks with proposed acceptance thresholds → `validation_report.csv` |

**Golden rule – no imputation of CSAT/NPS:** missing survey values are never filled (no 0 / mean / median / model / carry-forward into the score).
A missing component simply drops out and the remaining weights are re-normalised; confidence reflects what was observed.

**Inputs:** `data/bronze/*.parquet` · **Outputs:** `data/silver/`, `data/gold/`, `reports/voice/` · **Period:** 1 Oct 2025 – 30 Sep 2026

## 0. Install packages (run once, then comment out)

In [ ]:
# %pip install pandas duckdb matplotlib scikit-learn scipy pyarrow

## 1. Configuration (change settings here only)

In [ ]:
import time, json
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import duckdb
import matplotlib.pyplot as plt
from scipy.stats import chi2_contingency, spearmanr
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score, mean_absolute_error, confusion_matrix
from IPython.display import display, Markdown

%matplotlib inline
pd.set_option("display.max_columns", 60); pd.set_option("display.width", 250)
pd.set_option("display.float_format", "{:,.4f}".format)

# ---- paths ----
BRONZE_DIR = Path("data/bronze")
SILVER_DIR = Path("data/silver")
GOLD_DIR = Path("data/gold")
OUT_DIR = Path("reports/voice")
WORK_DB = OUT_DIR / "voice_work.duckdb"
TEXT_PRED_PATH = SILVER_DIR / "text_sentiment_predictions.parquet"  # optional plug-in: interaction_id, text_sentiment_pred (-1..+1)
HEALTH_PATH = GOLD_DIR / "behavioral_health_scores.parquet"         # optional: customer_id, mth, health_score (0-100)

WINDOW_START, WINDOW_END = "2025-10-01", "2026-10-01"   # end exclusive

# ---- scoring (non-imputed) ----
WEIGHTS = {"text": 0.6, "csat": 0.2, "nps": 0.2}  # base weights; only OBSERVED components are used, then re-normalised
SURVEY_CAP = 0.30          # max combined CSAT+NPS base weight (limits double counting; corr(CSAT,NPS) ~0.67)
CONFLICT_GAP = 0.8         # text vs survey disagree in sign AND differ by >= this -> conflict (confidence downgraded)
LOOKBACK_MONTHS = 3        # three-state context flags (detractor_seen / surveyed_no_detractor / not_surveyed)
SCORE_VERSION = "voice_v0.1"

# ---- analysis / validation ----
MIN_N_INDEX = 100          # minimum responses to publish an NPS index (otherwise suppressed)
MCAR_SPREAD_PP = 1.5       # response-rate spread (pp) across groups considered negligible
TEST_PCT = 20              # % of customers in the hold-out (customer-grouped split)
TEMPLATE_HOLDOUT_PCT = 20  # % of distinct texts held out (tests generalisation to unseen wording)
OUTCOME_HORIZON = 3        # months ahead for future outcomes (inactivity, next NPS)
ISSUE_TYPES = ("complaint", "billing_dispute", "cancellation_request", "technical_issue")
OPEN_STATUS = ("unresolved", "escalated", "pending")
PROFILE_COLS = ("loyalty_tier", "customer_segment", "region", "age", "gender", "income_band", "preferred_channel")

# proposed acceptance thresholds - agree with business before sign-off
THRESHOLDS = {
    "V0_split_overlap":             ("==", 0),
    "V1_macro_f1_customer_holdout": (">=", 0.80),
    "V1_macro_f1_template_holdout": (">=", 0.70),
    "V1_spearman_text_vs_gt":       (">=", 0.70),
    "V2_spearman_text_vs_nps":      (">=", 0.50),
    "V2_auc_detractor":             (">=", 0.75),
    "V3_delta_auc_text_plus_nps":   (">=", 0.00),
    "V4_delta_auc_default_vs_text": (">=", 0.00),
    "V5_decile_rho_inactivity":     ("<=", -0.80),
    "V5_decile_rho_leave_nps_out":  (">=", 0.80),
    "V6_error_monotonic_by_band":   ("==", 1),
    "V7_max_nps_response_spread_pp": ("<=", MCAR_SPREAD_PP),
    "V7_text_mean_diff_nps_vs_none": ("<=", 0.02),
    "V8_max_tier_auc_gap":          ("<=", 0.05),
    "V9_max_psi":                   ("<=", 0.10),
}

DUCKDB_THREADS, DUCKDB_MEMORY = 4, "4GB"   # lower to 2 / "2GB" if the kernel crashes
REBUILD = True                             # False = reuse tables already in WORK_DB
SHOW_ROWS = 12

## 2. Connection and helpers

In [ ]:
for d in (SILVER_DIR, GOLD_DIR, OUT_DIR):
    d.mkdir(parents=True, exist_ok=True)
try:
    con.close()
except NameError:
    pass
con = duckdb.connect(str(WORK_DB))
con.execute(f"SET threads={DUCKDB_THREADS}")
con.execute(f"SET memory_limit='{DUCKDB_MEMORY}'")
con.execute(f"SET temp_directory='{(OUT_DIR / 'duckdb_tmp').as_posix()}'")
con.execute("SET preserve_insertion_order=false")

FINDINGS, RESULTS = [], {}


def q(sql):
    return con.sql(sql).df()


def sql_list(v):
    return "(" + ", ".join(f"'{x}'" for x in v) + ")"


def save(df, name, show=SHOW_ROWS):
    df.to_csv(OUT_DIR / f"{name}.csv", index=False)
    print(f"--- {name} ({len(df):,} rows) -> {OUT_DIR / (name + '.csv')}")
    if show:
        display(df.head(show))
    return df


def flag(tag, msg):
    FINDINGS.append((tag, msg))
    print(f"  [FLAG] {msg}")


def table_exists(name):
    return bool(q(f"SELECT count(*) FROM information_schema.tables WHERE table_name='{name}'").iloc[0, 0])


def cramers_v(ct):
    chi2, p, _, _ = chi2_contingency(ct)
    n = ct.values.sum()
    k = min(ct.shape) - 1
    return float(np.sqrt(chi2 / (n * k))) if k > 0 else 0.0, float(p)


print("DuckDB", duckdb.__version__, "| pandas", pd.__version__, "| outputs ->", OUT_DIR.resolve())

## 3. Register sources (typed views; invalid survey values are set to NULL, never repaired)

In [ ]:
def src(t):
    f, d = BRONZE_DIR / f"{t}.parquet", BRONZE_DIR / t
    if f.exists():
        return f"read_parquet('{f.as_posix()}')"
    if d.exists():
        return f"read_parquet('{(d / '**' / '*.parquet').as_posix()}')"
    return None

for t in ("customers", "interactions", "engagements", "transactions"):
    s = src(t)
    if s is None and t in ("customers", "interactions"):
        raise FileNotFoundError(f"{t} parquet not found in {BRONZE_DIR.resolve()}")
    if s:
        con.execute(f"CREATE OR REPLACE VIEW raw_{t} AS SELECT * FROM {s}")
HAS_TX, HAS_ENG = src("transactions") is not None, src("engagements") is not None
print("transactions:", HAS_TX, "| engagements:", HAS_ENG)

con.execute("""
CREATE OR REPLACE TABLE cust1 AS
SELECT customer_id, TRY_CAST(signup_date AS DATE) signup, loyalty_tier, customer_segment, region, preferred_channel,
       (upper(trim(CAST(marketing_consent AS VARCHAR))) IN ('Y','YES','TRUE','1')) marketing_consent
FROM raw_customers
QUALIFY row_number() OVER (PARTITION BY customer_id ORDER BY signup_date) = 1
""")

con.execute(f"""
CREATE OR REPLACE VIEW itx AS
SELECT interaction_id, customer_id, ts, CAST(date_trunc('month', ts) AS DATE) mth,
       channel, interaction_type, resolution_status,
       (upper(trim(CAST(first_contact_resolution AS VARCHAR))) IN ('Y','YES','TRUE','1')) fcr,
       csat_raw, nps_raw,
       CASE WHEN csat_raw BETWEEN 1 AND 5 AND csat_raw = round(csat_raw) THEN csat_raw END csat,   -- invalid -> NULL (not repaired)
       CASE WHEN nps_raw BETWEEN 0 AND 10 AND nps_raw = round(nps_raw) THEN nps_raw END nps,
       CAST(interaction_text AS VARCHAR) txt, sentiment_label, TRY_CAST(sentiment_score_gt AS DOUBLE) sscore
FROM (SELECT *, TRY_CAST(interaction_ts AS TIMESTAMP) ts,
             TRY_CAST(csat_score AS DOUBLE) csat_raw, TRY_CAST(nps_score AS DOUBLE) nps_raw
      FROM raw_interactions)
WHERE ts >= TIMESTAMP '{WINDOW_START}' AND ts < TIMESTAMP '{WINDOW_END}'
""")

tx_sql = (f"""SELECT customer_id, CAST(date_trunc('month', TRY_CAST(transaction_ts AS TIMESTAMP)) AS DATE) mth, count(*) n_tx
              FROM raw_transactions GROUP BY ALL""" if HAS_TX else
          "SELECT CAST(NULL AS VARCHAR) customer_id, CAST(NULL AS DATE) mth, 0 n_tx WHERE false")
eng_sql = (f"""SELECT customer_id, CAST(date_trunc('month', TRY_CAST(event_ts AS TIMESTAMP)) AS DATE) mth, count(*) n_eng
               FROM raw_engagements GROUP BY ALL""" if HAS_ENG else
           "SELECT CAST(NULL AS VARCHAR) customer_id, CAST(NULL AS DATE) mth, 0 n_eng WHERE false")
con.execute(f"""
CREATE OR REPLACE TABLE act_m AS
WITH t AS ({tx_sql}), e AS ({eng_sql}),
     i AS (SELECT customer_id, mth, count(*) n_int FROM itx GROUP BY ALL),
     te AS (SELECT coalesce(t.customer_id, e.customer_id) customer_id, coalesce(t.mth, e.mth) mth,
                   coalesce(n_tx, 0) n_tx, coalesce(n_eng, 0) n_eng
            FROM t FULL JOIN e ON t.customer_id = e.customer_id AND t.mth = e.mth)
SELECT coalesce(te.customer_id, i.customer_id) customer_id, coalesce(te.mth, i.mth) mth,
       coalesce(n_tx, 0) n_tx, coalesce(n_eng, 0) n_eng, coalesce(n_int, 0) n_int
FROM te FULL JOIN i ON te.customer_id = i.customer_id AND te.mth = i.mth
WHERE coalesce(te.mth, i.mth) >= DATE '{WINDOW_START}' AND coalesce(te.mth, i.mth) < DATE '{WINDOW_END}'
""")
display(q("SELECT (SELECT count(*) FROM cust1) customers, (SELECT count(*) FROM itx) interactions, (SELECT count(*) FROM act_m) active_customer_months"))

# Part A – CSAT / NPS deep profile
Scales: **CSAT 1–5**, **NPS 0–10** (Promoter 9–10, Passive 7–8, Detractor 0–6). All statistics use observed responses only.

### A1. Coverage at interaction level and the independence check

In [ ]:
a1 = q("""
SELECT count(*) n_interactions,
       count(csat) n_csat, round(100.0*count(csat)/count(*), 2) csat_pct,
       count(nps)  n_nps,  round(100.0*count(nps)/count(*), 2)  nps_pct,
       count_if(csat IS NOT NULL AND nps IS NOT NULL) n_both,
       round(100.0*count_if(csat IS NOT NULL AND nps IS NOT NULL)/count(*), 2) both_pct,
       count_if(csat IS NULL AND nps IS NOT NULL) n_nps_only,
       count_if(csat IS NOT NULL AND nps IS NULL) n_csat_only,
       round(100.0*count_if(nps IS NOT NULL AND csat IS NOT NULL)/nullif(count(csat),0), 2) p_nps_given_csat,
       round(100.0*count_if(nps IS NOT NULL AND csat IS NULL)/nullif(count_if(csat IS NULL),0), 2) p_nps_given_no_csat
FROM itx""")
a1["expected_both_pct_if_independent"] = (a1.csat_pct * a1.nps_pct / 100).round(2)
a1["observed_vs_expected"] = (a1.both_pct / a1.expected_both_pct_if_independent).round(3)
save(a1, "A1_coverage_interaction_level")
RESULTS["nps_pct"], RESULTS["csat_pct"] = a1.nps_pct.iloc[0], a1.csat_pct.iloc[0]
r = a1.observed_vs_expected.iloc[0]
flag("A1", f"NPS on {a1.nps_pct.iloc[0]}% / CSAT on {a1.csat_pct.iloc[0]}% of interactions; both-together ratio "
           f"{r} vs 1.0 if independent -> {'surveys issued independently' if abs(r-1) < 0.05 else 'surveys are linked - check design'}")

### A2. Distributions

In [ ]:
csat_d = save(q("""SELECT CAST(csat AS INT) score, count(*) n, round(100.0*count(*)/sum(count(*)) OVER (), 2) pct
                   FROM itx WHERE csat IS NOT NULL GROUP BY 1 ORDER BY 1"""), "A2_csat_distribution")
nps_d = save(q("""SELECT CAST(nps AS INT) score, count(*) n, round(100.0*count(*)/sum(count(*)) OVER (), 2) pct,
                         CASE WHEN nps >= 9 THEN 'promoter' WHEN nps >= 7 THEN 'passive' ELSE 'detractor' END category
                  FROM itx WHERE nps IS NOT NULL GROUP BY 1, 4 ORDER BY 1"""), "A2_nps_distribution")
cat = nps_d.groupby("category").pct.sum()
flag("A2", f"NPS mix: promoters {cat.get('promoter',0):.1f}%, passives {cat.get('passive',0):.1f}%, detractors "
           f"{cat.get('detractor',0):.1f}% -> NPS index {cat.get('promoter',0)-cat.get('detractor',0):+.1f}")

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].bar(csat_d.score.astype(str), csat_d.pct, color="#4472C4"); ax[0].set_title("CSAT distribution (%)")
colors = nps_d.category.map({"detractor": "#C00000", "passive": "#BFBFBF", "promoter": "#00B050"})
ax[1].bar(nps_d.score.astype(str), nps_d.pct, color=colors); ax[1].set_title("NPS distribution (%) – red detractor / grey passive / green promoter")
plt.tight_layout(); plt.savefig(OUT_DIR / "A2_distributions.png", dpi=120); plt.show()

### A3. Missingness mechanism – is the response rate the same everywhere?
If response rates do not depend on channel, type, sentiment, resolution, month or tier, responses are **missing (close to) completely at random (MCAR)** on observables → survey averages are unbiased and *presence of a survey carries no information* (never use “has NPS” as a feature).
With 1M rows tiny differences are “significant”, so use **effect size (Cramér’s V; <0.02 negligible)** and the **spread in percentage points**.

In [ ]:
DIMS = ["channel", "interaction_type", "resolution_status", "sentiment_label", "fcr", "mth",
        "loyalty_tier", "customer_segment", "region"]
con.execute("CREATE OR REPLACE TEMP VIEW itx_c AS SELECT i.*, c.loyalty_tier, c.customer_segment, c.region FROM itx i LEFT JOIN cust1 c USING (customer_id)")
rows, frames = [], []
for d in DIMS:
    t = q(f"""SELECT CAST({d} AS VARCHAR) level, count(*) n, count(nps) n_nps, count(csat) n_csat,
                     round(100.0*count(nps)/count(*), 2) nps_resp_pct, round(100.0*count(csat)/count(*), 2) csat_resp_pct,
                     round(avg(nps), 3) avg_nps, round(avg(csat), 3) avg_csat,
                     CASE WHEN count(nps) >= {MIN_N_INDEX}
                          THEN round(100.0*(count_if(nps >= 9) - count_if(nps <= 6))/count(nps), 1) END nps_index
              FROM itx_c GROUP BY 1 ORDER BY 1""")
    frames.append(t.assign(dimension=d))
    out = {"dimension": d, "levels": len(t)}
    for s, col in (("nps", "n_nps"), ("csat", "n_csat")):
        ct = pd.DataFrame({"yes": t[col], "no": t.n - t[col]})
        v, p = cramers_v(ct)
        out.update({f"{s}_spread_pp": round(t[f"{s}_resp_pct"].max() - t[f"{s}_resp_pct"].min(), 2),
                    f"{s}_cramers_v": round(v, 4), f"{s}_p_value": p})
    rows.append(out)
save(pd.concat(frames)[["dimension", "level", "n", "n_nps", "nps_resp_pct", "n_csat", "csat_resp_pct",
                        "avg_nps", "avg_csat", "nps_index"]], "A3_response_rate_by_dimension", show=0)
MCAR = save(pd.DataFrame(rows), "A3_missingness_summary", show=20)
RESULTS["max_nps_spread_pp"] = MCAR.nps_spread_pp.max()
bad = MCAR[(MCAR.nps_spread_pp > MCAR_SPREAD_PP) | (MCAR.nps_cramers_v >= 0.02)]
if len(bad):
    flag("A3", "NPS response rate varies with: " + ", ".join(bad.dimension) + " -> responses NOT random; weight/report by group")
else:
    flag("A3", f"NPS response rate flat across all {len(DIMS)} dimensions (max spread {MCAR.nps_spread_pp.max()} pp, "
               f"max V {MCAR.nps_cramers_v.max()}) -> MCAR on observables; survey presence is uninformative")

### A4. Customer-level coverage, responses per customer, gap between responses

In [ ]:
a4 = q("""
WITH c AS (SELECT customer_id, count(*) n_int, count(nps) n_nps, count(csat) n_csat FROM itx GROUP BY 1)
SELECT (SELECT count(*) FROM cust1) customers,
       count(*) customers_with_interaction,
       count_if(n_csat > 0) customers_with_csat, count_if(n_nps > 0) customers_with_nps,
       round(100.0*count_if(n_csat > 0)/(SELECT count(*) FROM cust1), 2) pct_all_customers_with_csat,
       round(100.0*count_if(n_nps > 0)/(SELECT count(*) FROM cust1), 2)  pct_all_customers_with_nps
FROM c""")
save(a4, "A4_customer_coverage")
per = save(q("""WITH c AS (SELECT customer_id, count(nps) k FROM itx GROUP BY 1 HAVING k > 0)
                SELECT k nps_responses_in_period, count(*) customers, round(100.0*count(*)/sum(count(*)) OVER (), 2) pct
                FROM c GROUP BY 1 ORDER BY 1"""), "A4_nps_responses_per_customer")
gaps = save(q("""WITH r AS (SELECT DISTINCT customer_id, mth FROM itx WHERE nps IS NOT NULL),
                      g AS (SELECT date_diff('month', lag(mth) OVER (PARTITION BY customer_id ORDER BY mth), mth) gap FROM r)
                 SELECT gap months_between_nps, count(*) n FROM g WHERE gap IS NOT NULL GROUP BY 1 ORDER BY 1"""),
            "A4_months_between_nps_responses")
one = per.loc[per.nps_responses_in_period == 1, "pct"].sum()
flag("A4", f"{a4.pct_all_customers_with_nps.iloc[0]}% of customers have any NPS in 12 months; {one:.1f}% of them only once "
           "-> NPS is a snapshot, not a monthly per-customer series")

### A5. Customer-month coverage (active = any transaction, engagement or interaction)

In [ ]:
con.execute("""CREATE OR REPLACE TEMP VIEW sm AS
               SELECT customer_id, mth, count(nps) n_nps, count(csat) n_csat FROM itx GROUP BY ALL""")
a5 = q("""
SELECT a.mth, count(*) active_cm, count_if(a.n_int > 0) cm_with_interaction,
       round(100.0*count_if(s.n_nps > 0)/count(*), 2) nps_pct_of_active,
       round(100.0*count_if(s.n_nps > 0)/nullif(count_if(a.n_int > 0), 0), 2) nps_pct_of_interaction_months,
       round(100.0*count_if(s.n_csat > 0)/count(*), 2) csat_pct_of_active,
       round(100.0*count_if(s.n_csat > 0)/nullif(count_if(a.n_int > 0), 0), 2) csat_pct_of_interaction_months,
       round(100.0*count_if(s.n_csat > 0 OR s.n_nps > 0)/count(*), 2) any_survey_pct_of_active
FROM act_m a LEFT JOIN sm s USING (customer_id, mth) GROUP BY 1 ORDER BY 1""")
save(a5, "A5_customer_month_coverage_by_month")
tot = q("""SELECT round(100.0*count_if(s.n_nps > 0)/count(*), 2) nps, round(100.0*count_if(s.n_csat > 0)/count(*), 2) csat,
                  round(100.0*count_if(a.n_int > 0)/count(*), 2) interaction
           FROM act_m a LEFT JOIN sm s USING (customer_id, mth)""").iloc[0]
RESULTS["nps_cm_pct"] = tot.nps
flag("A5", f"Of active customer-months: {tot.interaction}% have an interaction, {tot.csat}% a CSAT, {tot.nps}% an NPS")

fig, ax = plt.subplots(figsize=(11, 4))
x = pd.to_datetime(a5.mth).dt.strftime("%Y-%m")
for c in ["nps_pct_of_active", "csat_pct_of_active", "any_survey_pct_of_active"]:
    ax.plot(x, a5[c], marker="o", label=c)
ax.set_ylabel("% of active customer-months"); ax.set_title("Survey coverage of active customer-months")
ax.tick_params(axis="x", rotation=45); ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.savefig(OUT_DIR / "A5_coverage_by_month.png", dpi=120); plt.show()

### A6. What drives NPS / CSAT? (interaction vs relationship)
Variance explained (R²) by interaction attributes and by the ground-truth text sentiment. High R² ⇒ NPS here is largely a **transactional (interaction-level) score**.

In [ ]:
for d in ["interaction_type", "resolution_status", "fcr", "sentiment_label"]:
    save(q(f"""SELECT CAST({d} AS VARCHAR) {d}, count(nps) n_nps, round(avg(nps), 2) avg_nps, round(avg(csat), 2) avg_csat,
                      round(100.0*count_if(nps >= 9)/nullif(count(nps), 0), 1) promoter_pct,
                      round(100.0*count_if(nps <= 6)/nullif(count(nps), 0), 1) detractor_pct,
                      CASE WHEN count(nps) >= {MIN_N_INDEX}
                           THEN round(100.0*(count_if(nps >= 9) - count_if(nps <= 6))/count(nps), 1) END nps_index
               FROM itx GROUP BY 1 ORDER BY avg_nps"""), f"A6_nps_by_{d}")

r6 = q("SELECT interaction_type, resolution_status, fcr, sentiment_label, sscore, nps, csat FROM itx WHERE nps IS NOT NULL")
var = r6.nps.var(ddof=0)
r2 = lambda g: 1 - ((r6.nps - r6.groupby(g).nps.transform("mean")) ** 2).mean() / var
rows = [{"explained_by": " + ".join(g), "r2": round(r2(g), 3)} for g in
        (["interaction_type"], ["resolution_status"], ["fcr"], ["sentiment_label"], ["interaction_type", "sentiment_label"],
         ["interaction_type", "sentiment_label", "resolution_status"])]
b = np.polyfit(r6.sscore, r6.nps, 1)
rows.append({"explained_by": "sscore (linear)", "r2": round(1 - (r6.nps - np.polyval(b, r6.sscore)).var(ddof=0) / var, 3)})
R2 = save(pd.DataFrame(rows), "A6_nps_variance_explained")
both = q("SELECT csat, nps, sscore FROM itx WHERE csat IS NOT NULL AND nps IS NOT NULL")
save(both.corr(method="spearman").round(3).reset_index(), "A6_spearman_csat_nps_sentiment")
RESULTS["corr_csat_nps"] = both.corr(method="spearman").loc["csat", "nps"]
flag("A6", f"Text sentiment explains {R2.r2.iloc[-1]:.0%} of NPS variance; Spearman(CSAT,NPS)={RESULTS['corr_csat_nps']:.2f} "
           "-> NPS is interaction-driven; CSAT and NPS overlap (cap combined survey weight)")

### A7. Within-customer consistency (is there a stable 'relationship' component?)

In [ ]:
a7 = []
for s in ("nps", "csat"):
    r = q(f"""WITH x AS (SELECT customer_id, {s} v FROM itx WHERE {s} IS NOT NULL),
                   c AS (SELECT customer_id, count(*) k, var_samp(v) wv, avg(v) m FROM x GROUP BY 1 HAVING k >= 2)   -- unbiased within-customer variance
              SELECT '{s}' metric, count(*) customers_with_2plus, round(avg(sqrt(wv)), 3) avg_within_sd,
                     (SELECT round(stddev_pop(v), 3) FROM x) overall_sd,
                     round(1 - avg(wv)/(SELECT var_pop(v) FROM x), 3) share_variance_between_customers
              FROM c""")
    a7.append(r)
a7 = save(pd.concat(a7), "A7_within_customer_consistency")
v = a7.set_index("metric").loc["nps", "share_variance_between_customers"]
flag("A7", f"~{v:.0%} of NPS variance is between customers (stable component); the rest is event-driven")

### A8. Mixed-signal texts – does NPS see what the text label misses?
Each text = *opener* (sets the GT label) + *issue clause*. If NPS differs across issue clauses **within the same opener/label**, NPS carries information the text label ignores → justifies a non-zero NPS weight.

In [ ]:
con.execute("""CREATE OR REPLACE TEMP VIEW itx_clause AS
               SELECT *, trim(split_part(txt, ',', 1)) opener, nullif(trim(split_part(txt, ',', 2)), '') issue_clause FROM itx""")
save(q("""SELECT opener, mode(sentiment_label) gt_label, count(DISTINCT txt) n_texts, count(*) n,
                 round(avg(nps), 2) avg_nps, round(avg(csat), 2) avg_csat, round(avg(sscore), 3) avg_gt_score
          FROM itx_clause GROUP BY 1 ORDER BY avg_nps"""), "A8_nps_by_opener", show=0)
a8 = q("""SELECT sentiment_label, issue_clause, count(*) n, count(nps) n_nps, round(avg(nps), 2) avg_nps,
                 round(avg(csat), 2) avg_csat, round(avg(sscore), 3) avg_gt_score
          FROM itx_clause WHERE issue_clause IS NOT NULL GROUP BY 1, 2 HAVING count(nps) >= 30 ORDER BY 1, avg_nps""")
save(a8, "A8_nps_by_issue_clause_within_label", show=0)
spread = a8.groupby("sentiment_label").agg(nps_range=("avg_nps", lambda s: s.max() - s.min()),
                                           gt_score_range=("avg_gt_score", lambda s: s.max() - s.min())).round(3).reset_index()
save(spread, "A8_spread_within_label")
pos = a8[a8.sentiment_label == "positive"]
if len(pos):
    fig, ax = plt.subplots(figsize=(10, 5.5))
    ax.barh(pos.issue_clause, pos.avg_nps, color="#4472C4")
    ax.set_xlim(pos.avg_nps.min() - .3, pos.avg_nps.max() + .1)
    ax.set_title("Avg NPS by issue clause – POSITIVE-labelled texts only"); ax.set_xlabel("avg NPS")
    plt.tight_layout(); plt.savefig(OUT_DIR / "A8_issue_clause_nps.png", dpi=120); plt.show()
    flag("A8", f"Within positive-labelled texts NPS ranges {pos.avg_nps.min():.2f}-{pos.avg_nps.max():.2f} by issue clause "
               "-> GT text label misses issue severity; NPS adds that signal")

### A9. Does NPS predict what happens next? (and beyond text sentiment)
Outcome = no transaction / engagement / interaction in the next 3 months. Responses in months with a full 3-month horizon only.

In [ ]:
con.execute(f"""
CREATE OR REPLACE TABLE a9 AS
WITH r AS (SELECT interaction_id, customer_id, mth, nps, sentiment_label, interaction_type,
                  CASE WHEN nps >= 9 THEN 'promoter' WHEN nps >= 7 THEN 'passive' ELSE 'detractor' END nps_cat
           FROM itx WHERE nps IS NOT NULL AND mth <= DATE '{WINDOW_END}' - INTERVAL {OUTCOME_HORIZON + 1} MONTH)
SELECT r.*, count(a.mth) active_months_next, coalesce(sum(a.n_int), 0) interactions_next
FROM r LEFT JOIN act_m a ON a.customer_id = r.customer_id AND a.mth > r.mth
                         AND a.mth <= r.mth + INTERVAL {OUTCOME_HORIZON} MONTH
GROUP BY ALL""")
a9 = save(q("""SELECT nps_cat, count(*) n, round(100.0*count_if(active_months_next = 0)/count(*), 2) pct_inactive_next,
                      round(avg(active_months_next), 3) avg_active_months_next, round(avg(interactions_next), 3) avg_interactions_next
               FROM a9 GROUP BY 1 ORDER BY pct_inactive_next DESC"""), "A9_nps_vs_future_activity")
save(q("""SELECT sentiment_label, nps_cat, count(*) n, round(100.0*count_if(active_months_next = 0)/count(*), 2) pct_inactive_next
          FROM a9 GROUP BY 1, 2 HAVING count(*) >= 500 ORDER BY 1, 2"""), "A9_nps_vs_future_activity_within_label", show=12)
d, p = a9.set_index("nps_cat").pct_inactive_next.get("detractor"), a9.set_index("nps_cat").pct_inactive_next.get("promoter")
flag("A9", f"Inactive in next {OUTCOME_HORIZON}m: detractors {d}% vs promoters {p}% (lift {d/p:.2f}x); "
           "gap persists within the same text label -> NPS adds predictive signal beyond text")

### A10. Segment / tier levels and the NPS index (suppressed below the minimum sample)

In [ ]:
for d in ("loyalty_tier", "customer_segment", "region"):
    save(q(f"""SELECT {d}, count(DISTINCT customer_id) FILTER (WHERE nps IS NOT NULL) customers_with_nps,
                      count(nps) n_nps, round(100.0*count(nps)/count(*), 2) nps_resp_pct,
                      round(avg(nps), 2) avg_nps, round(avg(csat), 2) avg_csat,
                      CASE WHEN count(nps) >= {MIN_N_INDEX}
                           THEN round(100.0*(count_if(nps >= 9) - count_if(nps <= 6))/count(nps), 1) END nps_index,
                      count(nps) < {MIN_N_INDEX} suppressed
               FROM itx_c GROUP BY 1 ORDER BY avg_nps"""), f"A10_nps_by_{d}")
t = pd.read_csv(OUT_DIR / "A10_nps_by_loyalty_tier.csv")
flag("A10", f"Avg NPS ranges {t.avg_nps.min()}-{t.avg_nps.max()} across loyalty tiers -> never adjust a customer's score by tier; "
            "compare against own-tier benchmarks in reporting")

### A11. Hygiene and survey timing

In [ ]:
a11 = q(f"""
SELECT count_if(csat_raw IS NOT NULL AND csat IS NULL) invalid_csat_set_null,
       count_if(nps_raw IS NOT NULL AND nps IS NULL) invalid_nps_set_null,
       count_if(nps IS NOT NULL AND resolution_status IN {sql_list(OPEN_STATUS)}) nps_on_open_issue,
       count_if(nps IS NOT NULL AND resolution_status = 'pending') nps_on_pending,
       count_if(csat IS NOT NULL AND resolution_status = 'pending') csat_on_pending,
       count(*) - count(DISTINCT interaction_id) duplicate_interaction_ids
FROM itx""")
save(a11, "A11_hygiene")
if a11.nps_on_pending.iloc[0] > 0:
    flag("A11", f"{int(a11.nps_on_pending.iloc[0]):,} NPS responses on PENDING interactions - confirm when the survey is sent "
                "(no survey timestamp in data)")

### Part A – findings

In [ ]:
with pd.option_context("display.max_colwidth", None):
    display(pd.DataFrame(FINDINGS, columns=["check", "finding"]))

# Part B – Text sentiment component (input to the Sentiment Score)
* **Customer-grouped split** (`hash(customer_id) % 100 < TEST_PCT` → test) is assigned once and stored in the data model, so every downstream metric is out-of-sample by customer.
* If `TEXT_PRED_PATH` exists (Sentiment Agent / LLM output: `interaction_id, text_sentiment_pred`), it is used as-is.
* Otherwise a **baseline** (TF-IDF + Ridge on `sentiment_score_gt`, trained on **train customers only**) is fitted. The GT columns are labels – they never enter the score directly.
* Texts are highly templated, so a second **template hold-out** (unseen texts) is also evaluated in Part E.

In [ ]:
con.execute(f"""CREATE OR REPLACE TABLE split_c AS
                SELECT customer_id, CASE WHEN hash(customer_id) % 100 < {TEST_PCT} THEN 'test' ELSE 'train' END split FROM cust1""")
display(q("SELECT split, count(*) customers FROM split_c GROUP BY 1"))

n_texts = q("SELECT count(DISTINCT txt) FROM itx").iloc[0, 0]
print(f"distinct texts: {n_texts:,}  (interactions: {q('SELECT count(*) FROM itx').iloc[0,0]:,})")
if n_texts < 5000:
    flag("B", f"Only {n_texts:,} distinct texts -> templated text; text metrics on random splits are optimistic (see V1 template hold-out)")

LABELS = ["negative", "neutral", "positive"]

def fit_text_model(train_df):
    """train_df: txt, n, y (mean GT score), lab. Returns (vectorizer, model, (lo, hi) label cut-offs)."""
    vec = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True, min_df=1)
    X = vec.fit_transform(train_df.txt)
    mdl = Ridge(alpha=1.0).fit(X, train_df.y, sample_weight=train_df.n)
    pred = np.clip(mdl.predict(X), -1, 1)
    best = (-1, (-0.2, 0.2))
    for lo in np.arange(-0.6, 0.001, 0.02):           # cut-offs tuned on TRAIN only
        for hi in np.arange(0.0, 0.601, 0.02):
            lab = np.where(pred < lo, "negative", np.where(pred > hi, "positive", "neutral"))
            f = f1_score(train_df.lab, lab, average="macro", sample_weight=train_df.n)
            if f > best[0]:
                best = (f, (round(lo, 2), round(hi, 2)))
    return vec, mdl, best[1]

def to_label(pred, cuts):
    lo, hi = cuts
    return np.where(pred < lo, "negative", np.where(pred > hi, "positive", "neutral"))

texts = q("SELECT DISTINCT txt FROM itx WHERE txt IS NOT NULL")
if TEXT_PRED_PATH.exists():
    TEXT_SOURCE = "plug-in"
    con.execute(f"""CREATE OR REPLACE TABLE text_pred_i AS
                    SELECT interaction_id, greatest(-1, least(1, CAST(text_sentiment_pred AS DOUBLE))) text_pred
                    FROM read_parquet('{TEXT_PRED_PATH.as_posix()}')""")
    tr = q("""SELECT p.text_pred, i.sentiment_label lab FROM text_pred_i p JOIN itx i USING (interaction_id)
              JOIN split_c s USING (customer_id) WHERE s.split = 'train'""")
    best = (-1, None)
    for lo in np.arange(-0.6, 0.001, 0.02):
        for hi in np.arange(0.0, 0.601, 0.02):
            f = f1_score(tr.lab, to_label(tr.text_pred.values, (lo, hi)), average="macro")
            if f > best[0]:
                best = (f, (round(lo, 2), round(hi, 2)))
    CUTS = best[1]
else:
    TEXT_SOURCE = "baseline TF-IDF+Ridge"
    train_t = q("""SELECT i.txt, count(*) n, avg(i.sscore) y, mode(i.sentiment_label) lab
                   FROM itx i JOIN split_c s USING (customer_id) WHERE s.split = 'train' AND i.txt IS NOT NULL GROUP BY 1""")
    VEC, MDL, CUTS = fit_text_model(train_t)
    texts["text_pred"] = np.clip(MDL.predict(VEC.transform(texts.txt)), -1, 1)
    texts["text_label_pred"] = to_label(texts.text_pred.values, CUTS)
    con.register("texts_df", texts)
    con.execute("CREATE OR REPLACE TABLE text_lookup AS SELECT * FROM texts_df")
    con.unregister("texts_df")
print("text source:", TEXT_SOURCE, "| label cut-offs (tuned on train):", CUTS)

# Part C – Non-imputed scoring specification
**Normalisation to −1…+1** (observed values only):

$$\text{CSAT}_{norm}=\frac{\text{CSAT}-3}{2}\qquad \text{NPS}_{norm}=\frac{\text{NPS}-5}{5}$$

**Observed-only weights for customer *c*, month *m*:** base weights $w_{text}=0.6,\ w_{csat}=0.2,\ w_{nps}=0.2$; a component gets weight only if it was observed that month.
If CSAT and NPS are both observed, their combined weight is capped at `SURVEY_CAP` (0.30) to avoid double counting.

$$S_{cm}=\frac{\sum_{k\in \text{observed}} w_k\,\bar{x}_k}{\sum_{k\in \text{observed}} w_k},\qquad k\in\{\text{text},\ \text{CSAT},\ \text{NPS}\}$$

| Situation | Effective weights | Score |
|---|---|---|
| Text only | text 1.00 | text average |
| Text + CSAT | text 0.75, CSAT 0.25 | weighted |
| Text + NPS | text 0.75, NPS 0.25 | weighted |
| Text + CSAT + NPS (cap 0.30) | text 0.667, CSAT 0.167, NPS 0.167 | weighted |
| Nothing observed | – | **NULL – `not_available` (never neutral 0)** |

**Not allowed:** filling missing CSAT/NPS with 0/mean/median/model, carrying last NPS forward into the score, using “has survey” as a feature, using tier/segment in the score.
**Allowed:** observed counts, three-state look-back context flags (`not_surveyed` is its own state), last observed NPS shown as *labelled context*.

**Conflict:** text and survey have opposite signs and differ by ≥ `CONFLICT_GAP` → flagged, confidence downgraded one level.

**Confidence band** – `high`: text + ≥1 survey, no conflict · `medium`: ≥2 interactions text-only, or text+survey with conflict, or survey without text · `low`: 1 interaction text-only · `not_available`: nothing observed.
**Confidence score (0–1)** = 0.5 × coverage (observed base weight ÷ total) + 0.3 × volume (min(n_text,3)/3) + 0.2 × agreement (1 − |text − survey|/2; 0.5 if no survey to verify).

**Voice actions (NPS triggers use the month’s *minimum* NPS):** cancellation + negative → `retention_save_desk`; detractor or CSAT ≤ 2 → `service_recovery_48h`; score < −0.3 (medium/high conf.) → `proactive_service_check`; promoter & score ≥ 0.5 → `advocacy_referral_invite` (needs marketing consent, else `thank_you_service_only`); passive → `nurture_offer` (needs consent); else `monitor`.

### C1. Python reference implementation (used for worked examples, unit tests and the weight grid in Part E)

In [ ]:
def composite(text_avg, n_text, csat_norm, n_csat, nps_norm, n_nps, w=None, cap=SURVEY_CAP):
    """Vectorised observed-only composite. Returns DataFrame: score, survey_score, w_text_eff, w_csat_eff, w_nps_eff."""
    w = w or WEIGHTS
    n_text, n_csat, n_nps = (np.asarray(x, dtype=float) for x in (n_text, n_csat, n_nps))
    t, c, n = (np.asarray(x, dtype=float) for x in (text_avg, csat_norm, nps_norm))
    wt = np.where(n_text > 0, w["text"], 0.0)
    wc = np.where(n_csat > 0, w["csat"], 0.0)
    wn = np.where(n_nps > 0, w["nps"], 0.0)
    cap = np.inf if cap is None else cap
    scale = np.where(wc + wn > cap, cap / np.where(wc + wn > 0, wc + wn, 1), 1.0)
    wc, wn = wc * scale, wn * scale
    wsum = wt + wc + wn
    num = np.where(wt > 0, wt * np.nan_to_num(t), 0) + np.where(wc > 0, wc * np.nan_to_num(c), 0) \
        + np.where(wn > 0, wn * np.nan_to_num(n), 0)        # nan_to_num only where weight is 0 -> term excluded, not imputed
    with np.errstate(invalid="ignore", divide="ignore"):
        score = np.where(wsum > 0, num / wsum, np.nan)
        ssum = wc + wn
        survey = np.where(ssum > 0, (np.where(wc > 0, wc * np.nan_to_num(c), 0) + np.where(wn > 0, wn * np.nan_to_num(n), 0)) / ssum, np.nan)
        return pd.DataFrame({"score": score, "survey_score": survey,
                             "w_text_eff": np.where(wsum > 0, wt / wsum, np.nan),
                             "w_csat_eff": np.where(wsum > 0, wc / wsum, np.nan),
                             "w_nps_eff": np.where(wsum > 0, wn / wsum, np.nan)})


def confidence(n_text, text_avg, survey_score, n_csat, n_nps, w=None):
    w = w or WEIGHTS
    n_text, n_csat, n_nps = (np.asarray(x, dtype=float) for x in (n_text, n_csat, n_nps))
    t, s = np.asarray(text_avg, dtype=float), np.asarray(survey_score, dtype=float)
    has_t, has_s = n_text > 0, ~np.isnan(s)
    conflict = has_t & has_s & (np.nan_to_num(t) * np.nan_to_num(s) < 0) & (np.abs(np.nan_to_num(t) - np.nan_to_num(s)) >= CONFLICT_GAP)
    band = np.select([~has_t & ~has_s, has_t & has_s & ~conflict, has_t & has_s & conflict, ~has_t & has_s, n_text >= 2],
                     ["not_available", "high", "medium", "medium", "medium"], default="low")
    cov = (np.where(has_t, w["text"], 0) + np.where(n_csat > 0, w["csat"], 0) + np.where(n_nps > 0, w["nps"], 0)) / sum(w.values())
    agree = np.where(has_t & has_s, 1 - np.abs(np.nan_to_num(t) - np.nan_to_num(s)) / 2, 0.5)
    score = np.where(band == "not_available", np.nan, 0.5 * cov + 0.3 * np.minimum(n_text, 3) / 3 + 0.2 * agree)
    return pd.DataFrame({"confidence_band": band, "confidence_score": np.round(score, 3), "conflict": conflict})


ex = pd.DataFrame([
    ("1 interaction, text only",            1, 0.40, np.nan, 0, np.nan, 0),
    ("3 interactions, text only",           3, -0.30, np.nan, 0, np.nan, 0),
    ("text + CSAT 2",                        1, -0.20, -0.5, 1, np.nan, 0),
    ("text + NPS 3 (detractor)",             2, 0.10, np.nan, 0, -0.4, 1),
    ("text + CSAT 5 + NPS 10 (cap applies)", 1, 0.60, 1.0, 1, 1.0, 1),
    ("conflict: positive text, NPS 1",       1, 0.65, np.nan, 0, -0.8, 1),
    ("no interaction, no survey",            0, np.nan, np.nan, 0, np.nan, 0),
], columns=["case", "n_text", "text_avg", "csat_norm", "n_csat", "nps_norm", "n_nps"])
cmp_ = composite(ex.text_avg, ex.n_text, ex.csat_norm, ex.n_csat, ex.nps_norm, ex.n_nps)
cf_ = confidence(ex.n_text, ex.text_avg, cmp_.survey_score, ex.n_csat, ex.n_nps)
display(pd.concat([ex, cmp_.round(3), cf_], axis=1))

# Part D – Customer-month data model
```
dim_customer (1 row / customer, CONTEXT ONLY)            fact_interaction_scored (1 row / interaction)       [silver]
  customer_id PK, signup, loyalty_tier, segment,  ──1:N──▶  interaction_id PK, customer_id, mth, split,
  region, preferred_channel, marketing_consent              type, resolution, fcr, text_pred, text_label_pred,
                                                            csat, nps, csat_norm, nps_norm, nps_category (observed only)
            │                                                          │ aggregate (no imputation)
            ▼                                                          ▼
customer_month_voice  (1 row / customer_id × mth; spine = every month from signup, Oct-25…Sep-26)       [gold]
  keys · context(only) · activity coverage · interaction evidence · survey evidence (NULL when not observed)
  · look-back context (3-state) · score components & effective weights · sentiment_score · confidence · drivers · voice action · lineage
            │
            ▼
agg_segment_nps_month (mth × dimension × level; NPS index suppressed when n < MIN_N_INDEX)                [gold]
```
**Design rules:** grain `customer_id × mth`; spine keeps *every* eligible month so coverage denominators are honest;
survey value columns are NULL when not observed (counts are 0); context columns never feed the score;
the next Behavioral Health table joins on the same key.

### D1. `fact_interaction_scored` (silver)

In [ ]:
if TEXT_SOURCE == "plug-in":
    text_join = "LEFT JOIN text_pred_i p USING (interaction_id)"
    text_cols = "p.text_pred"
else:
    text_join = "LEFT JOIN text_lookup p USING (txt)"
    text_cols = "p.text_pred"
lo, hi = CUTS
con.execute(f"""
CREATE OR REPLACE TABLE fis AS
SELECT i.interaction_id, i.customer_id, i.ts, i.mth, s.split,
       i.channel, i.interaction_type, i.resolution_status, i.fcr,
       {text_cols} text_pred,
       CASE WHEN {text_cols} < {lo} THEN 'negative' WHEN {text_cols} > {hi} THEN 'positive'
            WHEN {text_cols} IS NOT NULL THEN 'neutral' END text_label_pred,
       i.csat, i.nps,
       (i.csat - 3) / 2.0 csat_norm,                 -- NULL stays NULL
       (i.nps - 5) / 5.0  nps_norm,
       CASE WHEN i.nps >= 9 THEN 'promoter' WHEN i.nps >= 7 THEN 'passive' WHEN i.nps >= 0 THEN 'detractor' END nps_category,
       i.sentiment_label gt_label, i.sscore gt_score   -- labels: evaluation only
FROM itx i JOIN split_c s USING (customer_id) {text_join}
""")
con.execute(f"COPY fis TO '{(SILVER_DIR / 'fact_interaction_scored.parquet').as_posix()}' (FORMAT parquet, COMPRESSION zstd)")
display(q("SELECT * FROM fis WHERE nps IS NOT NULL LIMIT 5"))
print("missing text_pred:", q("SELECT count_if(text_pred IS NULL) FROM fis").iloc[0, 0])

### D2. Build `customer_month_voice` (gold) – scoring, confidence, drivers and actions in SQL

In [ ]:
WT, WC, WN = WEIGHTS["text"], WEIGHTS["csat"], WEIGHTS["nps"]
WTOT = WT + WC + WN
COUNT_COLS = ["n_text", "n_text_negative", "n_issue_contacts", "n_cancel_req", "n_open_issue", "n_csat", "n_csat_low",
              "n_nps", "n_promoter", "n_passive", "n_detractor", "n_both_same_interaction", "n_nps_on_open_issue"]

SCORE_SQL = f"""
    CASE WHEN w_sum > 0 THEN (CASE WHEN n_text > 0 THEN w_t * text_avg ELSE 0 END
                            + CASE WHEN n_csat > 0 THEN w_c * csat_norm_avg ELSE 0 END
                            + CASE WHEN n_nps  > 0 THEN w_n * nps_norm_avg  ELSE 0 END) / w_sum END"""

t0 = time.time()
con.execute(f"""
CREATE OR REPLACE TABLE cm_voice AS
WITH months AS (SELECT CAST(range AS DATE) mth FROM range(DATE '{WINDOW_START}', DATE '{WINDOW_END}', INTERVAL 1 MONTH)),
spine AS (SELECT c.customer_id, m.mth FROM cust1 c CROSS JOIN months m
          WHERE c.signup IS NULL OR m.mth >= date_trunc('month', c.signup)),
iv AS (
  SELECT customer_id, mth,
         count(*) n_int, count(text_pred) n_text, avg(text_pred) text_avg, min(text_pred) text_min,
         count_if(text_label_pred = 'negative') n_text_negative,
         count_if(interaction_type IN {sql_list(ISSUE_TYPES)}) n_issue_contacts,
         count_if(interaction_type = 'cancellation_request') n_cancel_req,
         count_if(resolution_status IN {sql_list(OPEN_STATUS)}) n_open_issue,
         count(csat) n_csat, avg(csat) csat_avg, min(csat) csat_min, avg(csat_norm) csat_norm_avg, count_if(csat <= 2) n_csat_low,
         count(nps) n_nps, avg(nps) nps_avg, min(nps) nps_min, max(nps) nps_max, avg(nps_norm) nps_norm_avg,
         arg_min(nps, ts) FILTER (WHERE nps IS NOT NULL) nps_first_in_month,
         arg_max(nps, ts) FILTER (WHERE nps IS NOT NULL) nps_last_in_month,
         count_if(nps >= 9) n_promoter, count_if(nps BETWEEN 7 AND 8) n_passive, count_if(nps <= 6) n_detractor,
         count_if(csat IS NOT NULL AND nps IS NOT NULL) n_both_same_interaction,
         count_if(nps IS NOT NULL AND resolution_status IN {sql_list(OPEN_STATUS)}) n_nps_on_open_issue,
         max(ts) last_interaction_ts
  FROM fis GROUP BY ALL),
b AS (
  SELECT s.customer_id, s.mth, sp.split,
         c.loyalty_tier, c.customer_segment, c.region, c.preferred_channel, c.marketing_consent,
         coalesce(a.n_tx, 0) n_tx, coalesce(a.n_eng, 0) n_eng, coalesce(iv.n_int, 0) n_int,
         {", ".join(f"coalesce(iv.{x}, 0) {x}" for x in COUNT_COLS)},          -- counts: 0 is a fact, not an imputation
         iv.text_avg, iv.text_min, iv.csat_avg, iv.csat_min, iv.csat_norm_avg,
         iv.nps_avg, iv.nps_min, iv.nps_max, iv.nps_norm_avg, iv.nps_first_in_month, iv.nps_last_in_month,
         iv.last_interaction_ts
  FROM spine s JOIN cust1 c USING (customer_id) JOIN split_c sp USING (customer_id)
  LEFT JOIN act_m a ON a.customer_id = s.customer_id AND a.mth = s.mth
  LEFT JOIN iv ON iv.customer_id = s.customer_id AND iv.mth = s.mth),
w AS (
  SELECT *,
         CASE WHEN n_text > 0 THEN {WT} ELSE 0 END w_text_raw,
         CASE WHEN n_csat > 0 THEN {WC} ELSE 0 END w_csat_raw,
         CASE WHEN n_nps  > 0 THEN {WN} ELSE 0 END w_nps_raw
  FROM b),
w2 AS (
  SELECT *, w_text_raw w_t,
         w_csat_raw * CASE WHEN w_csat_raw + w_nps_raw > {SURVEY_CAP} THEN {SURVEY_CAP} / (w_csat_raw + w_nps_raw) ELSE 1 END w_c,
         w_nps_raw  * CASE WHEN w_csat_raw + w_nps_raw > {SURVEY_CAP} THEN {SURVEY_CAP} / (w_csat_raw + w_nps_raw) ELSE 1 END w_n
  FROM w),
sc AS (
  SELECT *, w_t + w_c + w_n w_sum FROM w2),
sc2 AS (
  SELECT *,
         {SCORE_SQL} sentiment_score,
         CASE WHEN w_c + w_n > 0 THEN (CASE WHEN n_csat > 0 THEN w_c * csat_norm_avg ELSE 0 END
                                     + CASE WHEN n_nps  > 0 THEN w_n * nps_norm_avg  ELSE 0 END) / (w_c + w_n) END survey_score
  FROM sc),
cf AS (
  SELECT *,
         coalesce(n_text > 0 AND survey_score IS NOT NULL AND text_avg * survey_score < 0
                  AND abs(text_avg - survey_score) >= {CONFLICT_GAP}, false) text_survey_conflict
  FROM sc2),
lb AS (
  SELECT *,
         max(CAST(n_detractor > 0 AS INT)) OVER l3 detr_l3, max(CAST(n_nps > 0 AS INT)) OVER l3 nps_l3,
         max(CAST(n_csat_low > 0 AS INT)) OVER l3 csat_low_l3, max(CAST(n_csat > 0 AS INT)) OVER l3 csat_l3,
         last_value(nps_last_in_month IGNORE NULLS) OVER upto last_nps_observed,
         last_value(CASE WHEN n_nps > 0 THEN mth END IGNORE NULLS) OVER upto last_nps_month,
         CAST(n_tx > 0 AS INT) + CAST(n_eng > 0 AS INT) + CAST(n_int > 0 AS INT) n_sources
  FROM cf
  WINDOW l3 AS (PARTITION BY customer_id ORDER BY mth ROWS BETWEEN {LOOKBACK_MONTHS - 1} PRECEDING AND CURRENT ROW),
         upto AS (PARTITION BY customer_id ORDER BY mth ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)),
fin AS (
  SELECT *,
         CASE WHEN sentiment_score IS NULL THEN 'not_available'
              WHEN n_text > 0 AND survey_score IS NOT NULL AND NOT text_survey_conflict THEN 'high'
              WHEN survey_score IS NOT NULL THEN 'medium'
              WHEN n_text >= 2 THEN 'medium' ELSE 'low' END confidence_band,
         CASE WHEN sentiment_score IS NOT NULL THEN round(
              0.5 * (w_text_raw + w_csat_raw + w_nps_raw) / {WTOT}
            + 0.3 * least(n_text, 3) / 3.0
            + 0.2 * CASE WHEN n_text > 0 AND survey_score IS NOT NULL THEN 1 - abs(text_avg - survey_score) / 2 ELSE 0.5 END, 3)
         END confidence_score,
         CASE WHEN w_sum > 0 AND n_text > 0 THEN w_t * text_avg / w_sum END contrib_text,
         CASE WHEN w_sum > 0 AND n_csat > 0 THEN w_c * csat_norm_avg / w_sum END contrib_csat,
         CASE WHEN w_sum > 0 AND n_nps  > 0 THEN w_n * nps_norm_avg  / w_sum END contrib_nps
  FROM lb)
SELECT
  -- keys
  customer_id, mth, split,
  -- context only (never used in the score)
  loyalty_tier, customer_segment, region, preferred_channel, marketing_consent,
  -- activity coverage
  n_tx, n_eng, n_int, n_tx > 0 has_tx, n_eng > 0 has_eng, n_int > 0 has_int,
  (n_tx > 0 OR n_eng > 0 OR n_int > 0) active, n_sources,
  coalesce(nullif(concat_ws('+', CASE WHEN n_tx > 0 THEN 'TX' END, CASE WHEN n_eng > 0 THEN 'ENG' END,
                                 CASE WHEN n_int > 0 THEN 'INT' END), ''), 'none') source_combo,
  -- interaction evidence
  n_text, round(text_avg, 4) text_avg, round(text_min, 4) text_min, n_text_negative,
  n_issue_contacts, n_cancel_req, n_open_issue, last_interaction_ts,
  -- survey evidence (observed only; NULL when not observed)
  n_csat > 0 has_csat, n_csat, csat_avg, csat_min, csat_norm_avg, n_csat_low,
  n_nps > 0 has_nps, n_nps, nps_avg, nps_min, nps_max, nps_norm_avg, nps_first_in_month, nps_last_in_month,
  n_promoter, n_passive, n_detractor,
  CASE WHEN n_detractor > 0 THEN 'detractor' WHEN n_passive > 0 THEN 'passive' WHEN n_promoter > 0 THEN 'promoter' END nps_category_month,
  n_both_same_interaction, n_nps_on_open_issue,
  CASE WHEN n_csat > 0 AND n_nps > 0 THEN 'csat+nps' WHEN n_csat > 0 THEN 'csat_only'
       WHEN n_nps > 0 THEN 'nps_only' ELSE 'none' END survey_coverage,
  coalesce(nullif(concat_ws('+', CASE WHEN n_text > 0 THEN 'text' END, CASE WHEN n_csat > 0 THEN 'csat' END,
                                 CASE WHEN n_nps > 0 THEN 'nps' END), ''), 'none') evidence_basis,
  -- look-back context (three-state; NOT used in the score)
  CASE WHEN detr_l3 = 1 THEN 'detractor_seen' WHEN nps_l3 = 1 THEN 'surveyed_no_detractor' ELSE 'not_surveyed' END nps_status_l{LOOKBACK_MONTHS}m,
  CASE WHEN csat_low_l3 = 1 THEN 'low_csat_seen' WHEN csat_l3 = 1 THEN 'surveyed_no_low_csat' ELSE 'not_surveyed' END csat_status_l{LOOKBACK_MONTHS}m,
  last_nps_observed, last_nps_month, date_diff('month', last_nps_month, mth) months_since_last_nps,
  -- score
  CASE WHEN w_sum > 0 THEN round(w_t / w_sum, 4) END w_text_eff,
  CASE WHEN w_sum > 0 THEN round(w_c / w_sum, 4) END w_csat_eff,
  CASE WHEN w_sum > 0 THEN round(w_n / w_sum, 4) END w_nps_eff,
  round(contrib_text, 4) contrib_text, round(contrib_csat, 4) contrib_csat, round(contrib_nps, 4) contrib_nps,
  round(survey_score, 4) survey_score, round(sentiment_score, 4) sentiment_score,
  CASE WHEN sentiment_score IS NULL THEN 'not_available' ELSE 'scored' END sentiment_status,
  CASE WHEN sentiment_score IS NULL THEN NULL
       WHEN contrib_nps IS NOT NULL AND contrib_nps <= least(coalesce(contrib_text, 9), coalesce(contrib_csat, 9)) THEN 'nps'
       WHEN contrib_csat IS NOT NULL AND contrib_csat <= coalesce(contrib_text, 9) THEN 'csat'
       ELSE 'text' END lowest_component,     -- 9 = sentinel for argmin only, not a value
  nullif(concat_ws('; ',
     CASE WHEN n_detractor > 0 THEN 'NPS detractor (min ' || CAST(nps_min AS INT) || ')' END,
     CASE WHEN n_detractor = 0 AND n_promoter > 0 THEN 'NPS promoter (' || CAST(nps_max AS INT) || ')' END,
     CASE WHEN n_detractor = 0 AND n_promoter = 0 AND n_passive > 0 THEN 'NPS passive' END,
     CASE WHEN n_csat_low > 0 THEN 'Low CSAT (min ' || CAST(csat_min AS INT) || ')' END,
     CASE WHEN csat_avg >= 4.5 THEN 'High CSAT' END,
     CASE WHEN n_text_negative > 0 THEN 'Negative text on ' || n_text_negative || ' of ' || n_text || ' interaction(s)' END,
     CASE WHEN n_cancel_req > 0 THEN 'Cancellation request' END,
     CASE WHEN n_open_issue > 0 THEN n_open_issue || ' open/escalated issue(s)' END,
     CASE WHEN text_survey_conflict THEN 'Text and survey disagree' END), '') drivers,
  -- confidence
  text_survey_conflict, confidence_band, confidence_score,
  -- voice action (combined later with Behavioral Health)
  CASE WHEN sentiment_score IS NULL THEN 'no_voice_signal'
       WHEN n_cancel_req > 0 AND (n_detractor > 0 OR sentiment_score < 0) THEN 'retention_save_desk'
       WHEN n_detractor > 0 OR n_csat_low > 0 THEN 'service_recovery_48h'
       WHEN sentiment_score < -0.3 AND confidence_band IN ('high', 'medium') THEN 'proactive_service_check'
       WHEN n_promoter > 0 AND sentiment_score >= 0.5 THEN CASE WHEN marketing_consent THEN 'advocacy_referral_invite' ELSE 'thank_you_service_only' END
       WHEN n_passive > 0 THEN CASE WHEN marketing_consent THEN 'nurture_offer' ELSE 'monitor_no_marketing_consent' END
       ELSE 'monitor' END voice_action,
  -- lineage
  '{SCORE_VERSION}' score_version, '{json.dumps(WEIGHTS)}|cap={SURVEY_CAP}|gap={CONFLICT_GAP}|text={TEXT_SOURCE}' score_params,
  current_timestamp built_at
FROM fin
""")
con.execute(f"COPY cm_voice TO '{(GOLD_DIR / 'customer_month_voice.parquet').as_posix()}' (FORMAT parquet, COMPRESSION zstd)")
print(f"cm_voice: {q('SELECT count(*) FROM cm_voice').iloc[0,0]:,} rows in {time.time()-t0:,.0f}s")
display(q("SELECT * FROM cm_voice WHERE has_nps AND has_csat LIMIT 3").T)

### D3. Data dictionary (exported)

In [ ]:
DESC = {
 "customer_id": ("key", "Customer identifier"), "mth": ("key", "Month (first day)"),
 "split": ("key", "Customer-grouped train/test assignment used by all validation"),
 "loyalty_tier": ("context only", "Never used in score; for benchmarks/fairness"), "customer_segment": ("context only", ""),
 "region": ("context only", ""), "preferred_channel": ("context only", "Action routing"),
 "marketing_consent": ("context only", "Gate for marketing actions"),
 "n_tx": ("activity", "Transactions in month"), "n_eng": ("activity", "Engagement events"), "n_int": ("activity", "Interactions"),
 "has_tx": ("activity", ""), "has_eng": ("activity", ""), "has_int": ("activity", ""),
 "active": ("activity", "Any source present"), "n_sources": ("activity", "0-3 behavioural sources"),
 "source_combo": ("activity", "TX/ENG/INT combination"),
 "n_text": ("interaction evidence", "Interactions with a text score"), "text_avg": ("interaction evidence", "Mean text sentiment (-1..1)"),
 "text_min": ("interaction evidence", "Most negative text"), "n_text_negative": ("interaction evidence", ""),
 "n_issue_contacts": ("interaction evidence", "Complaint/billing/cancel/technical"), "n_cancel_req": ("interaction evidence", ""),
 "n_open_issue": ("interaction evidence", "Unresolved/escalated/pending"), "last_interaction_ts": ("interaction evidence", ""),
 "has_csat": ("survey evidence", "Observed flag - NOT a model feature"), "n_csat": ("survey evidence", "Responses (0 if none)"),
 "csat_avg": ("survey evidence", "NULL if not observed"), "csat_min": ("survey evidence", "NULL if not observed"),
 "csat_norm_avg": ("survey evidence", "(CSAT-3)/2, NULL if not observed"), "n_csat_low": ("survey evidence", "CSAT <= 2"),
 "has_nps": ("survey evidence", "Observed flag - NOT a model feature"), "n_nps": ("survey evidence", "Responses (0 if none)"),
 "nps_avg": ("survey evidence", "NULL if not observed"), "nps_min": ("survey evidence", "Used for triggers"),
 "nps_max": ("survey evidence", ""), "nps_norm_avg": ("survey evidence", "(NPS-5)/5, NULL if not observed"),
 "nps_first_in_month": ("survey evidence", "Earliest response (validation outcome)"), "nps_last_in_month": ("survey evidence", ""),
 "n_promoter": ("survey evidence", ""), "n_passive": ("survey evidence", ""), "n_detractor": ("survey evidence", ""),
 "nps_category_month": ("survey evidence", "Worst category in month; NULL if not surveyed"),
 "n_both_same_interaction": ("survey evidence", "CSAT and NPS on same interaction"),
 "n_nps_on_open_issue": ("survey evidence", "Survey timing check"),
 "survey_coverage": ("coverage", "csat+nps / csat_only / nps_only / none"), "evidence_basis": ("coverage", "Components used in score"),
 f"nps_status_l{LOOKBACK_MONTHS}m": ("look-back context", "detractor_seen / surveyed_no_detractor / not_surveyed - not in score"),
 f"csat_status_l{LOOKBACK_MONTHS}m": ("look-back context", "low_csat_seen / surveyed_no_low_csat / not_surveyed - not in score"),
 "last_nps_observed": ("look-back context", "Labelled context only - never carried into score"),
 "last_nps_month": ("look-back context", ""), "months_since_last_nps": ("look-back context", ""),
 "w_text_eff": ("score", "Effective weight after re-normalisation"), "w_csat_eff": ("score", ""), "w_nps_eff": ("score", ""),
 "contrib_text": ("score", "Contribution to score"), "contrib_csat": ("score", ""), "contrib_nps": ("score", ""),
 "survey_score": ("score", "Weighted CSAT/NPS (-1..1), NULL if none"),
 "sentiment_score": ("score", "Observed-only composite (-1..1); NULL = not available"),
 "sentiment_status": ("score", "scored / not_available"), "lowest_component": ("score", "Most negative component"),
 "drivers": ("score", "Plain-English reasons"), "text_survey_conflict": ("confidence", "Sign disagreement >= CONFLICT_GAP"),
 "confidence_band": ("confidence", "high / medium / low / not_available"), "confidence_score": ("confidence", "0-1"),
 "voice_action": ("action", "NPS/CSAT-driven next action"), "score_version": ("lineage", ""), "score_params": ("lineage", ""),
 "built_at": ("lineage", ""),
}
schema = q("DESCRIBE cm_voice")[["column_name", "column_type"]]
schema["group"] = schema.column_name.map(lambda c: DESC.get(c, ("UNDOCUMENTED", ""))[0])
schema["description"] = schema.column_name.map(lambda c: DESC.get(c, ("", ""))[1])
save(schema, "D3_data_dictionary_customer_month_voice", show=0)
und = schema[schema.group == "UNDOCUMENTED"]
print("undocumented columns:", list(und.column_name) or "none")
display(schema.groupby("group").column_name.apply(", ".join).reset_index())

### D4. Imputation guard tests (must all PASS)

In [ ]:
pyref = q("""SELECT n_text, text_avg, n_csat, csat_norm_avg, n_nps, nps_norm_avg, sentiment_score, survey_score,
                    confidence_band, confidence_score
             FROM cm_voice USING SAMPLE reservoir(200000 ROWS) REPEATABLE (7)""")
ref = composite(pyref.text_avg, pyref.n_text, pyref.csat_norm_avg, pyref.n_csat, pyref.nps_norm_avg, pyref.n_nps)
refc = confidence(pyref.n_text, pyref.text_avg, ref.survey_score, pyref.n_csat, pyref.n_nps)
diff = np.nanmax(np.abs(ref.score.values - pyref.sentiment_score.astype(float).values))
same_null = (np.isnan(ref.score.values) == pyref.sentiment_score.isna().values).all()

tests = {
 "NPS values NULL when no response":  "SELECT count(*) FROM cm_voice WHERE n_nps = 0 AND (nps_avg IS NOT NULL OR nps_norm_avg IS NOT NULL OR nps_min IS NOT NULL)",
 "CSAT values NULL when no response": "SELECT count(*) FROM cm_voice WHERE n_csat = 0 AND (csat_avg IS NOT NULL OR csat_norm_avg IS NOT NULL)",
 "NPS weight 0 when not observed":    "SELECT count(*) FROM cm_voice WHERE n_nps = 0 AND coalesce(w_nps_eff, 0) <> 0",
 "CSAT weight 0 when not observed":   "SELECT count(*) FROM cm_voice WHERE n_csat = 0 AND coalesce(w_csat_eff, 0) <> 0",
 "No score without evidence":         "SELECT count(*) FROM cm_voice WHERE n_text = 0 AND n_csat = 0 AND n_nps = 0 AND sentiment_score IS NOT NULL",
 "not_available <=> NULL score":      "SELECT count(*) FROM cm_voice WHERE (sentiment_score IS NULL) <> (confidence_band = 'not_available')",
 "Score within [-1, 1]":              "SELECT count(*) FROM cm_voice WHERE sentiment_score < -1 OR sentiment_score > 1",
 "Effective weights sum to 1":        "SELECT count(*) FROM cm_voice WHERE sentiment_score IS NOT NULL AND abs(w_text_eff + w_csat_eff + w_nps_eff - 1) > 1e-3",
 "Survey cap respected":              f"SELECT count(*) FROM cm_voice WHERE w_csat_eff + w_nps_eff > {SURVEY_CAP}/({WEIGHTS['text']}+{SURVEY_CAP}) + 1e-3 AND n_text > 0",
}
rows = [{"test": k, "violations": int(q(v).iloc[0, 0])} for k, v in tests.items()]
rows.append({"test": "SQL score == Python reference (200k sample)", "violations": int(not (diff < 1e-3 and same_null))})
rows.append({"test": "SQL confidence band == Python reference", "violations": int((refc.confidence_band.values != pyref.confidence_band.values).sum())})
rows.append({"test": "No profile column in score formula", "violations": sum(c in SCORE_SQL for c in PROFILE_COLS)})
rows.append({"test": "No look-back / last-NPS field in score formula",
             "violations": sum(c in SCORE_SQL for c in ("last_nps", "nps_status", "csat_status", "has_nps", "has_csat"))})
guard = pd.DataFrame(rows); guard["result"] = np.where(guard.violations == 0, "PASS", "FAIL")
save(guard, "D4_imputation_guard_tests", show=20)
assert (guard.result == "PASS").all(), "Imputation guard failed - fix before using the scores"

### D5. Coverage and confidence of the data model

In [ ]:
d5 = q("""
SELECT mth, count(*) eligible_cm, count_if(active) active_cm,
       round(100.0*count_if(has_int)/nullif(count_if(active), 0), 2) pct_active_with_interaction,
       round(100.0*count_if(has_csat)/nullif(count_if(active), 0), 2) pct_active_with_csat,
       round(100.0*count_if(has_nps)/nullif(count_if(active), 0), 2) pct_active_with_nps,
       round(100.0*count_if(sentiment_status = 'scored')/nullif(count_if(active), 0), 2) pct_active_scored,
       count_if(confidence_band = 'high') high, count_if(confidence_band = 'medium') medium,
       count_if(confidence_band = 'low') low
FROM cm_voice GROUP BY 1 ORDER BY 1""")
save(d5, "D5_coverage_confidence_by_month")
save(q("""SELECT evidence_basis, confidence_band, count(*) n, round(100.0*count(*)/sum(count(*)) OVER (), 2) pct_of_scored,
                 round(avg(sentiment_score), 3) avg_score, round(avg(confidence_score), 3) avg_conf
          FROM cm_voice WHERE sentiment_status = 'scored' GROUP BY 1, 2 ORDER BY n DESC"""), "D5_evidence_by_confidence")
save(q(f"""SELECT nps_status_l{LOOKBACK_MONTHS}m, count(*) n, round(100.0*count(*)/sum(count(*)) OVER (), 2) pct
           FROM cm_voice WHERE active GROUP BY 1 ORDER BY n DESC"""), "D5_nps_lookback_status")
save(q("""SELECT voice_action, count(*) n, round(100.0*count(*)/sum(count(*)) OVER (), 2) pct
          FROM cm_voice WHERE sentiment_status = 'scored' GROUP BY 1 ORDER BY n DESC"""), "D5_voice_actions")

fig, ax = plt.subplots(figsize=(11, 4))
x = pd.to_datetime(d5.mth).dt.strftime("%Y-%m")
ax.bar(x, d5.high, label="high", color="#00B050"); ax.bar(x, d5.medium, bottom=d5.high, label="medium", color="#FFC000")
ax.bar(x, d5.low, bottom=d5.high + d5.medium, label="low", color="#C00000")
ax.set_title("Scored customer-months by confidence band"); ax.legend(); ax.tick_params(axis="x", rotation=45)
plt.tight_layout(); plt.savefig(OUT_DIR / "D5_confidence_by_month.png", dpi=120); plt.show()

### D6. Segment NPS aggregate (gold) – index suppressed below the minimum sample

In [ ]:
parts = []
for d in ("ALL", "loyalty_tier", "customer_segment", "region"):
    lvl = "'ALL'" if d == "ALL" else f"CAST({d} AS VARCHAR)"
    parts.append(f"""
      SELECT mth, '{d}' dimension, {lvl} level, count_if(active) active_customers,
             sum(n_nps) n_responses, count_if(has_nps) customers_responding,
             round(100.0*count_if(has_nps)/nullif(count_if(active), 0), 2) response_pct_of_active,
             round(sum(nps_avg * n_nps)/nullif(sum(n_nps), 0), 3) avg_nps,
             round(100.0*sum(n_promoter)/nullif(sum(n_nps), 0), 2) promoter_pct,
             round(100.0*sum(n_detractor)/nullif(sum(n_nps), 0), 2) detractor_pct,
             CASE WHEN sum(n_nps) >= {MIN_N_INDEX} THEN round(100.0*(sum(n_promoter) - sum(n_detractor))/sum(n_nps), 1) END nps_index,
             sum(n_nps) < {MIN_N_INDEX} suppressed
      FROM cm_voice GROUP BY ALL""")
con.execute("CREATE OR REPLACE TABLE agg_nps AS " + " UNION ALL ".join(parts))
con.execute(f"COPY agg_nps TO '{(GOLD_DIR / 'agg_segment_nps_month.parquet').as_posix()}' (FORMAT parquet, COMPRESSION zstd)")
save(q("SELECT * FROM agg_nps WHERE dimension IN ('ALL', 'loyalty_tier') ORDER BY dimension, level, mth"), "D6_agg_segment_nps_month", show=12)

# Part E – Validation framework (NPS & sentiment)
**Principles:** (1) customer-grouped hold-out – every metric reported on **test customers**; (2) **no circularity** – a score that contains NPS is never validated against the same NPS (use *leave-NPS-out* scores or **future** outcomes); (3) report **n and coverage** with every metric; (4) report by **loyalty tier** to show fairness; (5) thresholds are *proposed* and must be agreed with the business.

| # | Check | Question it answers | Data | Metric(s) | Proposed pass |
|---|---|---|---|---|---|
| V0 | Split integrity | Is the hold-out clean? | all | customers/rows by split, overlap = 0 | overlap 0 |
| V1 | Text vs ground truth | Is the text component accurate? | test interactions; + unseen-template hold-out | MAE, Spearman, accuracy, macro-F1, weighted-F1 | macro-F1 ≥ 0.80 (cust.), ≥ 0.70 (template); ρ ≥ 0.70 |
| V2 | Text vs NPS/CSAT | Does text agree with what customers *say*? | test interactions with NPS | Spearman, detractor AUC, promoter AUC, 3-class agreement | ρ ≥ 0.50; AUC ≥ 0.75 |
| V3 | NPS incremental value | Does NPS add signal beyond text? | test NPS responses, future 3m | AUC(text) vs AUC(text+NPS) for future inactivity | Δ ≥ 0 |
| V4 | Weight tuning & ablation | Are weights justified? | survey months, future outcomes | grid on train → test; AUC inactivity, ρ with next NPS | default ≥ text-only |
| V5 | Calibration / monotonicity | Do higher scores mean better outcomes? | test scored months | decile ρ vs future inactivity; leave-NPS-out decile ρ vs NPS | ρ ≤ −0.80; ρ ≥ 0.80 |
| V6 | Confidence validity | Is confidence meaningful? | test months with NPS (leave-NPS-out) | error by band (low ≥ medium ≥ high) | monotonic |
| V7 | Missingness (MCAR) | Is non-imputation safe? | all interactions | response-rate spread; text mean with vs without NPS | ≤ 1.5 pp; ≤ 0.02 |
| V8 | Tier fairness | Does it work equally across tiers? | test, by loyalty tier | detractor AUC gap vs overall (with 95% bootstrap CI) | ≤ 0.05 |
| V9 | Stability | Is it stable month to month? | all scored months | PSI vs reference month; lag-1 within-customer ρ | PSI ≤ 0.10 |
| V10 | Health cross-check (optional) | Does low Health precede detractors/inactivity? | Health table if present | detractor & inactivity rate by Health decile | monotonic |

In [ ]:
H = OUTCOME_HORIZON
con.execute(f"""
CREATE OR REPLACE TABLE val_cm AS
WITH o AS (
  SELECT customer_id, mth, split, loyalty_tier, n_int, n_text, text_avg, n_csat, csat_norm_avg, n_nps, nps_norm_avg, nps_avg,
         sentiment_score, sentiment_status, confidence_band, confidence_score,
         CASE WHEN mth <= DATE '{WINDOW_END}' - INTERVAL {H + 1} MONTH THEN
              1 - max(CAST(active AS INT)) OVER (PARTITION BY customer_id ORDER BY mth ROWS BETWEEN 1 FOLLOWING AND {H} FOLLOWING) END y_inactive_next,
         first_value(nps_first_in_month IGNORE NULLS) OVER (PARTITION BY customer_id ORDER BY mth
                                                            ROWS BETWEEN 1 FOLLOWING AND {H} FOLLOWING) y_next_nps
  FROM cm_voice)
SELECT * FROM o WHERE sentiment_status = 'scored'""")
V = q("SELECT * FROM val_cm")
print(f"validation customer-months (scored): {len(V):,}")
REPORT = []

def rec(check, metric, value, n=None, key=None, note=""):
    op, thr = THRESHOLDS.get(key, (None, None)) if key else (None, None)
    if op is None or value is None or (isinstance(value, float) and np.isnan(value)):
        status = "INFO"
    else:
        status = "PASS" if {">=": value >= thr, "<=": value <= thr, "==": value == thr}[op] else "FAIL"
    REPORT.append({"check": check, "metric": metric, "value": None if value is None else round(float(value), 4),
                   "n": n, "threshold": f"{op} {thr}" if op else "", "status": status, "note": note})

def auc(y, s):
    y, s = np.asarray(y), np.asarray(s)
    m = ~np.isnan(s.astype(float))
    return roc_auc_score(y[m], s[m]) if len(np.unique(y[m])) == 2 else np.nan

### V0. Split integrity

In [ ]:
v0 = q("""SELECT split, count(DISTINCT customer_id) customers, count(*) interactions, count(nps) nps_responses
          FROM fis GROUP BY 1""")
display(v0)
overlap = q("SELECT count(*) FROM (SELECT customer_id FROM fis GROUP BY 1 HAVING count(DISTINCT split) > 1)").iloc[0, 0]
rec("V0", "customers in both splits", overlap, key="V0_split_overlap")

### V1. Text component vs ground truth (customer hold-out and unseen-template hold-out)

In [ ]:
t1 = q("SELECT text_pred, text_label_pred, gt_label, gt_score FROM fis WHERE split = 'test' AND text_pred IS NOT NULL")
rec("V1", "MAE text vs GT score (customer hold-out)", mean_absolute_error(t1.gt_score, t1.text_pred), len(t1))
rec("V1", "Spearman text vs GT score", spearmanr(t1.text_pred, t1.gt_score)[0], len(t1), "V1_spearman_text_vs_gt")
rec("V1", "Accuracy label", accuracy_score(t1.gt_label, t1.text_label_pred), len(t1))
rec("V1", "Macro-F1 label (customer hold-out)", f1_score(t1.gt_label, t1.text_label_pred, average="macro"), len(t1),
    "V1_macro_f1_customer_holdout")
rec("V1", "Weighted-F1 label", f1_score(t1.gt_label, t1.text_label_pred, average="weighted"), len(t1))
display(pd.crosstab(t1.gt_label, t1.text_label_pred, margins=True))

if TEXT_SOURCE != "plug-in":
    allt = q(f"""SELECT i.txt, s.split, count(*) n, avg(i.sscore) y, mode(i.sentiment_label) lab,
                        hash(i.txt) % 100 < {TEMPLATE_HOLDOUT_PCT} held_out
                 FROM itx i JOIN split_c s USING (customer_id) GROUP BY 1, 2""")
    tr = allt[(allt.split == "train") & ~allt.held_out]
    te = allt[(allt.split == "test") & allt.held_out]
    vec_h, mdl_h, cuts_h = fit_text_model(tr)
    ph = np.clip(mdl_h.predict(vec_h.transform(te.txt)), -1, 1)
    rec("V1", "Macro-F1 label (unseen-template hold-out)",
        f1_score(te.lab, to_label(ph, cuts_h), average="macro", sample_weight=te.n), int(te.n.sum()),
        "V1_macro_f1_template_holdout", f"{len(te)} unseen texts")
    rec("V1", "Spearman (unseen-template hold-out)", spearmanr(ph, te.y)[0], int(te.n.sum()))
else:
    rec("V1", "Unseen-template hold-out", None, note="not applicable for plug-in predictions - evaluate in the Sentiment Agent pipeline")

### V2. Text vs what customers say (NPS / CSAT) – external validity

In [ ]:
t2 = q("SELECT text_pred, text_label_pred, nps, csat, nps_category, loyalty_tier FROM fis f JOIN cust1 USING (customer_id) "
       "WHERE split = 'test' AND nps IS NOT NULL AND text_pred IS NOT NULL")
rec("V2", "Spearman text vs NPS", spearmanr(t2.text_pred, t2.nps)[0], len(t2), "V2_spearman_text_vs_nps")
rec("V2", "AUC detractor (lower text = detractor)", auc((t2.nps <= 6).astype(int), -t2.text_pred), len(t2), "V2_auc_detractor")
rec("V2", "AUC promoter", auc((t2.nps >= 9).astype(int), t2.text_pred), len(t2))
t2c = q("SELECT text_pred, csat FROM fis WHERE split = 'test' AND csat IS NOT NULL AND text_pred IS NOT NULL")
rec("V2", "Spearman text vs CSAT", spearmanr(t2c.text_pred, t2c.csat)[0], len(t2c))
mapping = {"negative": "detractor", "neutral": "passive", "positive": "promoter"}
pred_cat = t2.text_label_pred.map(mapping)
rec("V2", "3-class agreement text label vs NPS category", (pred_cat == t2.nps_category).mean(), len(t2),
    note="NPS passive band is narrow - expect moderate agreement")
display(pd.crosstab(t2.nps_category, t2.text_label_pred, normalize="index").round(3))

### V3. Does NPS add predictive value beyond text? (future inactivity, customer-grouped)

In [ ]:
t3 = q(f"""SELECT f.split, f.text_pred, f.nps_norm, v.y_inactive_next
           FROM fis f JOIN val_cm v USING (customer_id, mth)
           WHERE f.nps IS NOT NULL AND f.text_pred IS NOT NULL AND v.y_inactive_next IS NOT NULL""")
tr, te = t3[t3.split == "train"], t3[t3.split == "test"]
res = {}
for name, cols in (("text", ["text_pred"]), ("text+nps", ["text_pred", "nps_norm"])):
    m = LogisticRegression().fit(tr[cols], tr.y_inactive_next)
    res[name] = roc_auc_score(te.y_inactive_next, m.predict_proba(te[cols])[:, 1])
rec("V3", "AUC future inactivity - text only", res["text"], len(te))
rec("V3", "AUC future inactivity - text + NPS", res["text+nps"], len(te))
rec("V3", "Delta AUC from adding NPS", res["text+nps"] - res["text"], len(te), "V3_delta_auc_text_plus_nps",
    "behavioural signals will carry most of the churn signal; this tests incremental voice value only")

### V4. Weight tuning & ablation (non-circular: judged on **future** outcomes)
Grid on **train** survey-months (text weight fixed 0.6, no cap) → best by $J = 0.5\,AUC_{inactivity} + 0.5\,\frac{\rho_{next\ NPS}+1}{2}$ → compared on **test** with *text-only* and the *default* config.

In [ ]:
S = V[(V.n_text > 0) & ((V.n_csat > 0) | (V.n_nps > 0)) & V.y_inactive_next.notna()].copy()

def evaluate(df, w, cap):
    s = composite(df.text_avg, df.n_text, df.csat_norm_avg, df.n_csat, df.nps_norm_avg, df.n_nps, w=w, cap=cap).score.values
    a = auc(df.y_inactive_next.astype(int), -s)
    m = df.y_next_nps.notna().values
    rho = spearmanr(s[m], df.y_next_nps.values[m])[0] if m.sum() > 30 else np.nan
    return a, rho, 0.5 * a + 0.5 * (np.nan_to_num(rho) + 1) / 2

grid = []
for wc in (0, .1, .2, .3):
    for wn in (0, .1, .2, .3, .4):
        w = {"text": .6, "csat": wc, "nps": wn}
        a, rho, j = evaluate(S[S.split == "train"], w, None)
        grid.append({"w_csat": wc, "w_nps": wn, "auc_inactivity": a, "rho_next_nps": rho, "J": j})
grid = save(pd.DataFrame(grid).sort_values("J", ascending=False), "V4_weight_grid_train", show=8)
best = grid.iloc[0]
cfgs = {"text_only": ({"text": 1, "csat": 0, "nps": 0}, None),
        "default": (WEIGHTS, SURVEY_CAP),
        "best_grid": ({"text": .6, "csat": best.w_csat, "nps": best.w_nps}, None)}
out = []
for k, (w, cap) in cfgs.items():
    a, rho, j = evaluate(S[S.split == "test"], w, cap)
    out.append({"config": k, "weights": json.dumps(w), "cap": cap, "auc_inactivity_test": a, "rho_next_nps_test": rho, "J_test": j})
v4 = save(pd.DataFrame(out), "V4_config_comparison_test")
d = v4.set_index("config")
rec("V4", "AUC inactivity: default - text-only (test)", d.loc["default", "auc_inactivity_test"] - d.loc["text_only", "auc_inactivity_test"],
    int((S.split == "test").sum()), "V4_delta_auc_default_vs_text")
rec("V4", "Spearman with next NPS: default (test)", d.loc["default", "rho_next_nps_test"])
rec("V4", "Best grid weights (train)", None, note=f"csat={best.w_csat}, nps={best.w_nps}; J_test={d.loc['best_grid','J_test']:.4f} "
                                                  f"vs default {d.loc['default','J_test']:.4f}")

### V5. Calibration and monotonicity

In [ ]:
T = V[(V.split == "test") & V.y_inactive_next.notna()].copy()
T["decile"] = pd.qcut(T.sentiment_score.rank(method="first"), 10, labels=False) + 1
dec = T.groupby("decile").agg(n=("sentiment_score", "size"), avg_score=("sentiment_score", "mean"),
                              inactive_next_rate=("y_inactive_next", "mean"), next_nps_mean=("y_next_nps", "mean")).reset_index()
save(dec, "V5_deciles_future_outcomes")
rho_d = spearmanr(dec.decile, dec.inactive_next_rate)[0]
rec("V5", "Decile rho: score vs future inactivity", rho_d, len(T), "V5_decile_rho_inactivity")

L = V[(V.split == "test") & (V.n_nps > 0) & (V.n_text > 0)].copy()
L["score_wo_nps"] = composite(L.text_avg, L.n_text, L.csat_norm_avg, L.n_csat, L.nps_norm_avg, np.zeros(len(L))).score.values
L["decile"] = pd.qcut(L.score_wo_nps.rank(method="first"), 10, labels=False) + 1
dl = L.groupby("decile").agg(n=("nps_avg", "size"), avg_score_wo_nps=("score_wo_nps", "mean"), avg_nps=("nps_avg", "mean"),
                             detractor_rate=("nps_avg", lambda s: (s <= 6).mean())).reset_index()
save(dl, "V5_leave_nps_out_deciles")
rec("V5", "Decile rho: leave-NPS-out score vs same-month NPS", spearmanr(dl.decile, dl.avg_nps)[0], len(L), "V5_decile_rho_leave_nps_out")

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(dec.decile, dec.inactive_next_rate * 100, marker="o"); ax[0].set_title(f"Future inactivity (next {H}m) by score decile")
ax[0].set_xlabel("score decile (1 = most negative)"); ax[0].set_ylabel("%")
ax[1].plot(dl.decile, dl.avg_nps, marker="o", color="#00B050"); ax[1].set_title("Observed NPS by leave-NPS-out score decile")
ax[1].set_xlabel("decile"); ax[1].set_ylabel("avg NPS")
for a in ax: a.grid(alpha=.3)
plt.tight_layout(); plt.savefig(OUT_DIR / "V5_calibration.png", dpi=120); plt.show()

### V6. Confidence validity (leave-NPS-out: does a higher band mean a smaller error vs observed NPS?)

In [ ]:
cw = composite(L.text_avg, L.n_text, L.csat_norm_avg, L.n_csat, L.nps_norm_avg, np.zeros(len(L)))
cb = confidence(L.n_text, L.text_avg, cw.survey_score, L.n_csat, np.zeros(len(L)))
L["band_wo_nps"], L["conf_wo_nps"] = cb.confidence_band.values, cb.confidence_score.values
L["abs_error"] = np.abs(cw.score.values - L.nps_norm_avg.values)
v6 = save(L.groupby("band_wo_nps").agg(n=("abs_error", "size"), mean_abs_error=("abs_error", "mean")).reset_index(), "V6_error_by_band")
e = v6.set_index("band_wo_nps").mean_abs_error
order = [b for b in ("low", "medium", "high") if b in e.index]
mono = int(all(e[order[i]] >= e[order[i + 1]] - 1e-9 for i in range(len(order) - 1)))
rec("V6", "Error decreases low -> medium -> high", mono, len(L), "V6_error_monotonic_by_band",
    " / ".join(f"{b}={e[b]:.3f}" for b in order) + " | if FAIL: more evidence does not bring the score closer to a single "
    "interaction-level NPS -> re-weight confidence towards survey presence / recency rather than interaction volume")
rec("V6", "Spearman(confidence_score, -error)", spearmanr(L.conf_wo_nps, -L.abs_error)[0], len(L))

### V7. Missingness – is non-imputation safe?

In [ ]:
rec("V7", "Max NPS response-rate spread across groups (pp)", RESULTS["max_nps_spread_pp"], None, "V7_max_nps_response_spread_pp")
t7 = q("SELECT nps IS NOT NULL has_nps, avg(text_pred) m, count(*) n FROM fis WHERE split = 'test' GROUP BY 1").set_index("has_nps")
rec("V7", "|mean text| with NPS vs without", abs(t7.loc[True, "m"] - t7.loc[False, "m"]), int(t7.n.sum()), "V7_text_mean_diff_nps_vs_none",
    "≈0 => surveyed interactions look like the rest")

### V8. Fairness by loyalty tier (context only – never in the score)

In [ ]:
rows = []
overall = auc((t2.nps <= 6).astype(int), -t2.text_pred)
rng = np.random.default_rng(42)
def boot_ci(y, s, reps=200):
    y, s = np.asarray(y), np.asarray(s); vals = []
    for _ in range(reps):
        i = rng.integers(0, len(y), len(y))
        if len(np.unique(y[i])) == 2:
            vals.append(roc_auc_score(y[i], s[i]))
    return np.percentile(vals, [2.5, 97.5])
for tier, g in t2.groupby("loyalty_tier"):
    vt = V[(V.split == "test") & (V.loyalty_tier == tier) & V.y_inactive_next.notna()]
    rows.append({"loyalty_tier": tier, "n_nps": len(g), "spearman_text_nps": spearmanr(g.text_pred, g.nps)[0],
                 "auc_detractor": auc((g.nps <= 6).astype(int), -g.text_pred),
                 "auc_ci_low": boot_ci((g.nps <= 6).astype(int), -g.text_pred)[0],
                 "auc_ci_high": boot_ci((g.nps <= 6).astype(int), -g.text_pred)[1],
                 "detractor_rate": (g.nps <= 6).mean(),
                 "auc_inactivity_score": auc(vt.y_inactive_next.astype(int), -vt.sentiment_score),
                 "avg_sentiment_score": vt.sentiment_score.mean(), "avg_nps": g.nps.mean()})
v8 = save(pd.DataFrame(rows), "V8_tier_fairness")
worst = v8.loc[(v8.auc_detractor - overall).abs().idxmax()]
rec("V8", "Max |detractor AUC tier - overall|", (v8.auc_detractor - overall).abs().max(), len(t2), "V8_max_tier_auc_gap",
    f"worst: {worst.loyalty_tier} AUC {worst.auc_detractor:.3f} (95% CI {worst.auc_ci_low:.3f}-{worst.auc_ci_high:.3f}, n={worst.n_nps}, "
    f"detractor rate {worst.detractor_rate:.1%}) vs overall {overall:.3f}")

### V9. Stability over time

In [ ]:
def psi(ref, cur, bins):
    e = np.histogram(ref, bins)[0] / len(ref); a = np.histogram(cur, bins)[0] / len(cur)
    e, a = np.clip(e, 1e-6, None), np.clip(a, 1e-6, None)
    return float(np.sum((a - e) * np.log(a / e)))

months = sorted(V.mth.unique()); ref_m = months[0]
ref = V.loc[V.mth == ref_m, "sentiment_score"].values
bins = np.unique(np.quantile(ref, np.linspace(0, 1, 11))); bins[0], bins[-1] = -1.0001, 1.0001
v9 = pd.DataFrame([{"mth": m, "n": int((V.mth == m).sum()), "mean_score": V.loc[V.mth == m, "sentiment_score"].mean(),
                    "psi_vs_ref": psi(ref, V.loc[V.mth == m, "sentiment_score"].values, bins)} for m in months])
save(v9, "V9_stability_psi")
rec("V9", f"Max PSI vs {pd.Timestamp(ref_m):%Y-%m}", v9.psi_vs_ref.max(), len(V), "V9_max_psi")
s = V[["customer_id", "mth", "sentiment_score"]].sort_values(["customer_id", "mth"])
s["prev"] = s.groupby("customer_id").sentiment_score.shift(); s["gap"] = s.groupby("customer_id").mth.diff().dt.days
c1 = s[(s.gap > 0) & (s.gap <= 31)]
rec("V9", "Lag-1 within-customer correlation (consecutive months)", c1.sentiment_score.corr(c1.prev), len(c1),
    note="low values = score is event-driven (expected for interaction-based voice)")

### V10. Behavioral Health cross-check (runs only if the Health table exists)

In [ ]:
if HEALTH_PATH.exists():
    con.execute(f"CREATE OR REPLACE VIEW health AS SELECT * FROM read_parquet('{HEALTH_PATH.as_posix()}')")
    h = q("""SELECT h.health_score, v.y_inactive_next, c.nps_category_month
             FROM health h JOIN val_cm v USING (customer_id, mth) JOIN cm_voice c USING (customer_id, mth)
             WHERE v.split = 'test' AND v.y_inactive_next IS NOT NULL""")
    h["decile"] = pd.qcut(h.health_score.rank(method="first"), 10, labels=False) + 1
    v10 = h.groupby("decile").agg(n=("health_score", "size"), inactive_next=("y_inactive_next", "mean"),
                                  detractor_rate=("nps_category_month", lambda s: (s == "detractor").sum() / max(s.notna().sum(), 1))).reset_index()
    save(v10, "V10_health_vs_nps")
    rec("V10", "Decile rho: health vs future inactivity", spearmanr(v10.decile, v10.inactive_next)[0], len(h))
else:
    rec("V10", "Health cross-check", None, note=f"skipped - {HEALTH_PATH} not found (run after the Behavioral Health model)")

## Validation report

In [ ]:
report = pd.DataFrame(REPORT)
report.to_csv(OUT_DIR / "validation_report.csv", index=False)
def colour(v):
    return {"PASS": "background-color:#C6EFCE", "FAIL": "background-color:#FFC7CE", "INFO": "background-color:#EDEDED"}.get(v, "")
sty = report.style.map if hasattr(report.style, "map") else report.style.applymap
display(sty(colour, subset=["status"]).format({"value": "{:.4f}"}, na_rep="-"))
print(report.status.value_counts().to_dict())

## Outputs and findings

In [ ]:
with pd.option_context("display.max_colwidth", None):
    display(pd.DataFrame(FINDINGS, columns=["check", "finding"]))
lines = ["# CSAT/NPS voice scoring - summary", "", f"Score version: {SCORE_VERSION} | weights {WEIGHTS} | cap {SURVEY_CAP} | text source: {TEXT_SOURCE}", "",
         "## Findings", ""] + [f"- **{a}** {b}" for a, b in FINDINGS] + ["", "## Validation", "",
         "| check | metric | value | threshold | status |", "|---|---|---|---|---|"] + \
        [f"| {r.check} | {r.metric} | {'' if pd.isna(r.value) else round(r.value, 4)} | {r.threshold} | {r.status} |" for r in report.itertuples()]
(OUT_DIR / "voice_summary.md").write_text("\n".join(lines), encoding="utf-8")
for p in [SILVER_DIR / "fact_interaction_scored.parquet", GOLD_DIR / "customer_month_voice.parquet",
          GOLD_DIR / "agg_segment_nps_month.parquet", OUT_DIR / "validation_report.csv", OUT_DIR / "voice_summary.md"]:
    print(f"{p}  ({p.stat().st_size/1e6:,.1f} MB)")

In [ ]:
con.close()   # release the DuckDB file lock